In [1]:
import csv
import sys
import pandas as pd
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from constants import IMG_OUTPUT_PATH, EMB_SPACES

PRECOMPUTED_KNN_PATH = f'{IMG_OUTPUT_PATH}/knn-binding-sites'
def read_imbalanced_kNN(k, emb_space_name, metric, do_average=True, mean_centering=False):
    results = {}
    with open(f'{PRECOMPUTED_KNN_PATH}/imbalanced/{metric}/{emb_space_name},k={k}{",mean_centered" if mean_centering else ""}.csv', 'r') as f:
        knn_results = csv.reader(f, delimiter=',')
        header = next(knn_results)
        for row in knn_results:
            feature, acc = row[0], float(row[1])
            class_name, number_of_points = feature.split(' ')[0], int(feature.split(' ')[-1][:-1])
            results[class_name] = (acc, number_of_points)
        
    if do_average:
        results = sum(float(a) * n for a, n in results.values()) / sum(n for _, n in results.values())
        return results
    else:
        return results
    

def read_kNN(k, emb_space_name, metric, do_average=True, mean_centering=False):
    results = {}
    with open(f'{PRECOMPUTED_KNN_PATH}/{metric}/k={k}{",mean_centered" if mean_centering else ""}.csv', 'r') as f:
        knn_results = csv.reader(f, delimiter=',')
        header = next(knn_results)
        emb_space_index = header.index(emb_space_name)
        for row in knn_results:
            feature, acc = row[0], float(row[emb_space_index])
            class_name, number_of_points = feature.split(' ')[0], int(feature.split(' ')[-1][:-1])
            results[class_name] = (acc, number_of_points)
        
    if do_average:
        results = sum(float(a) * n for a, n in results.values()) / sum(n for _, n in results.values())
        return results
    else:
        return results

print(read_kNN(3, 'ESM2', 'euclidean'))
print(read_kNN(5, 'ESM2', 'euclidean'))
print(read_kNN(10, 'ESM2', 'euclidean'))
print(read_kNN(50, 'ESM2', 'euclidean'))
print(read_kNN(100, 'ESM2', 'euclidean'))
print(read_kNN(200, 'ESM2', 'euclidean'))

print(read_imbalanced_kNN(3, 'ESM2', 'euclidean'))
print(read_imbalanced_kNN(5, 'ESM2', 'euclidean'))
print(read_imbalanced_kNN(10, 'ESM2', 'euclidean'))
print(read_imbalanced_kNN(50, 'ESM2', 'euclidean'))
print(read_imbalanced_kNN(100, 'ESM2', 'euclidean'))


0.589807530807068
0.5720602601676179
0.543164172965191
0.4771124479407682
0.45747647693968846
0.441333359041596
0.9255120350009433
0.9194724402818927
0.9084769948951519
0.8751233818492983
0.861996554668087


In [2]:
import plotly.express as px
def plot_kNN_alignment_scores(fig):
    fig.update_layout(
            yaxis_title="Mean KNN feature<br>alignment score",
            template="plotly_white",
            font={"family": "Arial", "color": "black", "size": 12},
            legend_title_text="Embedding models",
            width=1200,
            height=500,
    )

    fig.update_traces(marker=dict(size=10), line=dict(width=4))

    for axis in fig.layout:
        if axis.startswith('xaxis'):
            fig.layout[axis].update(
                        showgrid=False,
                        linecolor='black',
                        linewidth=3,
                        ticks='outside',
                        tickwidth=2,
                        tickcolor='black',
                        ticklen=6,
                        tickformat=".0f"
                    )
        if axis.startswith('yaxis'):
            fig.layout[axis].update(
                showgrid=False,
                linecolor='black',
                linewidth=3,
                ticks='outside',
                tickwidth=2,
                tickcolor='black',
                ticklen=6,
                tickformat=".2f",
                dtick=0.01
            )    

    for annotation in fig.layout.annotations:
        if '=' in annotation.text:
            annotation.text = annotation.text.split('=')[1]
                
    fig.update_layout(
        title=dict(
            x=0,
            xanchor='left'
        ),
        legend=dict(
            orientation='h',
            x=0.48,
            y=1.05,
            xanchor='center',
            yanchor='bottom'
        ),
        margin=dict(t=120)
    )
    return fig


In [5]:
import pickle
import plotly.colors as pc
import plotly.express as px

from constants import EMBEDDINGS_PATH, EMB_SPACES

Ks = [5, 10, 20, 30, 50, 100]
MEAN_CENTERING = False

# read precomputed kNN scores
collected_results = []
for metric in ['euclidean', 'cosine', 'cityblock']:
    for k in Ks:
        for emb_space in EMB_SPACES:
            collected_results.append((k, emb_space[0], read_imbalanced_kNN(k, emb_space[0], metric, mean_centering=MEAN_CENTERING), metric))

df = pd.DataFrame(collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric'])

# read precomputed MLP performance
performance_results = {}
for emb_space in EMB_SPACES:
    with open(f'{IMG_OUTPUT_PATH}/performance/train2-epochs=10/{emb_space[0]}_torch.pkl', 'rb') as f:
        performance_results[emb_space[0]] = pickle.load(f)

# combine kNN scores with MLP performance for legend text and color coding
new_collected_results = []
for result in collected_results:
    emb_space = result[1]
    performance = float(performance_results[emb_space]['test_auprc_macro' if 'test_auprc_macro' in performance_results[emb_space] else 'test_auprc'])
    legend_text = f'{performance:.3f} ({emb_space})'
    new_collected_results.append(result + (legend_text, performance))
df = pd.DataFrame(new_collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric', 'Legend text', 'Performance'])

# get line color acccording to the MLP performance 
sorted_performance = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=True)
sorted_performance_descending = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=False)
n_bins = len(sorted_performance)
low, high = 0.3, 1.0
color_scale = pc.sample_colorscale(
        "Reds",
        [low + (high - low) * i / (n_bins - 1) for i in range(n_bins)],
        colortype='rgb'
        )
label_to_color ={}
for label, color in zip(sorted_performance['Legend text'].tolist(), color_scale):
    label_to_color[label] = color

# plot
fig = px.line(
    df,
    x="k",
    y="kNN score",
    color="Legend text",
    color_discrete_map=label_to_color,
    facet_col="metric",
    markers=True,
    category_orders={"Legend text": sorted_performance_descending['Legend text'].tolist()},
    title=f'IMBALANCED{", mean centered" if MEAN_CENTERING else ""}'
)
fig = plot_kNN_alignment_scores(fig)
fig.show()

### Reformat plot
Change layout to fit with other plots.

In [8]:
import sys, os
import pandas as pd
import plotly
import plotly.express as px

sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/figs')
import fig_utils

output_dir = '/home/unix/vkrhk/EmmaEmb/EmmaEmb/img/figs'
filename = f'{output_dir}/3a.csv'

if os.path.isfile(filename):
    df_for_fig_utils = pd.read_csv(filename)
else:   
    df_for_fig_utils = df[['k', 'Embedding Space', 'kNN score', 'metric']]
    df_for_fig_utils.rename(columns={"kNN score": "Fraction"}, inplace=True)
    df_for_fig_utils[df_for_fig_utils[['Embedding Space']] == 'ANKH'] = 'Ankh'
    df_for_fig_utils.to_csv('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/figs/tmp/3a.csv', index=False)

fig = fig_utils.plot_knn_alignment_lines_model(df_for_fig_utils, 
                                         facet_col='metric', 
                                         color_col='Embedding Space', 
                                         y_title="Mean KNN feature alignment score *",
                                         facet_aliases={'cosine': 'Cosine distance', 'euclidean': 'Euclidean distance', 'cityblock': 'Manhattan distance'},
                                         facet_order=['cosine', 'cityblock', 'euclidean'],
                                         title='D. Ligand binding site (three-class)'
                                         )

import time
_kaleido_warmed = False
def write_pdf(fig, path, width=600, height=600):
    global _kaleido_warmed
    if not _kaleido_warmed:
        dummy = px.scatter(x=[0], y=[0])
        output_dir = '/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/figs'
        print("Warming up Kaleido...")
        dummy.write_image(output_dir + "_kaleido_warmup.pdf", format="pdf")
        print("Kaleido warmup complete.")
        time.sleep(2)
        _kaleido_warmed = True
    fig.write_image(path, format="pdf", width=width, height=height)
write_pdf(fig, f'{output_dir}/fig3,supplementary,no-center,imbalanced.pdf',
          width=1200, height=500)
fig.show()

Warming up Kaleido...
Kaleido warmup complete.


# Balanced case
Compute kNN score on balanced dataset (same number of binding vs. cryptic binding vs. non-binding). 
Train & evaluate models on balanced dataset too.

In [6]:
import pickle
import plotly.colors as pc
import plotly.express as px

from constants import EMB_SPACES

Ks = [5, 10, 20, 30, 50, 100]
MEAN_CENTERING = True

# read precomputed kNN scores
collected_results = []
for metric in ['euclidean', 'cosine', 'cityblock']:
    for k in Ks:
        for emb_space in EMB_SPACES:
            collected_results.append((k, emb_space[0], read_kNN(k, emb_space[0], metric, mean_centering=MEAN_CENTERING), metric))

df = pd.DataFrame(collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric'])

# read precomputed MLP performance
performance_results = {}
for emb_space in EMB_SPACES:
    with open(f'{IMG_OUTPUT_PATH}/performance/train,balanced,e=10/{emb_space[0]}_torch.pkl', 'rb') as f:
        performance_results[emb_space[0]] = pickle.load(f)

# combine kNN scores with MLP performance for legend text and color coding
new_collected_results = []
for result in collected_results:
    emb_space = result[1]
    performance = float(performance_results[emb_space]['test_auprc_macro'])
    legend_text = f'{performance:.3f} ({emb_space})'
    new_collected_results.append(result + (legend_text, performance))
df = pd.DataFrame(new_collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric', 'Legend text', 'Performance'])

# get line color acccording to the MLP performance 
sorted_performance = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=True)
sorted_performance_descending = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=False)
n_bins = len(sorted_performance)
low, high = 0.3, 1.0
color_scale = pc.sample_colorscale(
        "Reds",
        [low + (high - low) * i / (n_bins - 1) for i in range(n_bins)],
        colortype='rgb'
        )
label_to_color ={}
for label, color in zip(sorted_performance['Legend text'].tolist(), color_scale):
    label_to_color[label] = color

# plot
fig = px.line(
    df,
    x="k",
    y="kNN score",
    color="Legend text",
    color_discrete_map=label_to_color,
    facet_col="metric",
    markers=True,
    category_orders={"Legend text": sorted_performance_descending['Legend text'].tolist()},
    title=f'BALANCED{", mean centered" if MEAN_CENTERING else ""}'
)
fig = plot_kNN_alignment_scores(fig)
fig.show()

### Reformat plot
Change layout to fit with other plots.

In [10]:
import sys, os
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/figs')
%reload_ext autoreload
%autoreload 2
import fig_utils
import json
import plotly.express as px
import pandas as pd

output_dir = '/home/unix/vkrhk/EmmaEmb/EmmaEmb/img/figs'
filename = f'{output_dir}/3b.csv'
ml_perf_path = f'{output_dir}/ml_performance.json'
model_order_path = f'{output_dir}/model_order.json'

if os.path.isfile(filename) and os.path.isfile(ml_perf_path) and os.path.isfile(model_order_path):
    df_for_fig_utils = pd.read_csv(filename)
    
    # Loading from JSON
    with open(ml_perf_path, 'r') as f:
        ml_performance = json.load(f)
    with open(model_order_path, 'r') as f:
        model_order = json.load(f)
else:   
    df_for_fig_utils = df[['k', 'Embedding Space', 'kNN score']][df['metric'] == 'cosine'].copy()
    df_for_fig_utils.rename(columns={"kNN score": "Fraction"}, inplace=True)
    
    ml_performance = {}
    for emb_space, performance in performance_results.items():
        ml_performance[emb_space] = performance['test_auprc_macro'] * 100

    model_order = [i[0] for i in sorted(ml_performance.items(), key=lambda item: item[1], reverse=True)]   

    # ANKH -> Ankh cleanup
    df_for_fig_utils['Embedding Space'] = df_for_fig_utils['Embedding Space'].replace('ANKH', 'Ankh')
    
    if 'ANKH' in ml_performance:
        ml_performance['Ankh'] = ml_performance.pop('ANKH')
    
    if 'ANKH' in model_order:
        model_order[model_order.index('ANKH')] = 'Ankh'

    # Saving to JSON (plain text)
    with open(ml_perf_path, 'w') as f:
        json.dump(ml_performance, f, indent=4)
    with open(model_order_path, 'w') as f:
        json.dump(model_order, f, indent=4)
        
    df_for_fig_utils.to_csv(filename, index=False)

fig = fig_utils.plot_model_ranking(
    df_for_fig_utils, ml_performance,
    model_order, 
    embedding_col='Embedding Space', 
    knn_y_title="Mean KNN alignment score *",
    ml_y_range=[40,60],
    ml_y_title='AUPRC',
    title='D. Ligand binding site (three-class)'
    )

import time
_kaleido_warmed = False
def write_pdf(fig, path, width=600, height=600):
    global _kaleido_warmed
    if not _kaleido_warmed:
        dummy = px.scatter(x=[0], y=[0])
        output_dir = '/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/figs'
        print("Warming up Kaleido...")
        dummy.write_image(output_dir + "_kaleido_warmup.pdf", format="pdf")
        print("Kaleido warmup complete.")
        time.sleep(2)
        _kaleido_warmed = True
    fig.write_image(path, format="pdf", width=width, height=height)
write_pdf(fig, f'{output_dir}/fig3,final,mean-centered,balanced.pdf',
          width=1000, height=500)
fig.show()

Warming up Kaleido...
Kaleido warmup complete.
